In [1]:
import os
import copernicusmarine

# Locally: interactive. On GCP: set these env vars from Secret Manager.
# COPERNICUSMARINE_SERVICE_USERNAME / COPERNICUSMARINE_SERVICE_PASSWORD
copernicusmarine.login()  # prompts once, stores creds locally

/Users/claytonpreston/Documents/NextWave/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO - 2026-11-18T11:31:53Z - Using existing credentials from /Users/claytonpreston/.copernicusmarine/.copernicusmarine-credentials. Use --force-overwrite combined with credentials (specified by arguments, netrc file or environment variables) to always overwrite.


True

In [ ]:
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import xarray as xr
import requests

TZ = "Europe/Athens"
# GREECE
# BBOX = dict(minimum_longitude=19.0, maximum_longitude=30.0, minimum_latitude=34.0, maximum_latitude=42.0)
BBOX = dict(minimum_longitude=18.0, maximum_longitude=30.5,minimum_latitude=33.5,  maximum_latitude=42.0)
# PELOPO
# BBOX = dict(minimum_longitude=20.5, maximum_longitude=23.7, minimum_latitude=36.0, maximum_latitude=38.5)

# facing = compass direction the beach looks toward; window = half-width of good swell angle
# SPOTS = {}
from spots import SPOTS

now = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0, tzinfo=None)
START, END = now - timedelta(hours=6), now + timedelta(days=7)

## spots

## Wave data
https://data.marine.copernicus.eu/product/MEDSEA_ANALYSISFORECAST_WAV_006_017/description

Coordinate reference system
WGS 84 (EPSG:4326)
Data assimilation
SWHWind speed
Update frequency
Daily – 06:00; 20:00
Format
NetCDF-4
Originating centre
HCMR (Greece)

In [4]:
cat = copernicusmarine.describe(product_id="MEDSEA_ANALYSISFORECAST_WAV_006_017")
for p in cat.products:
    for d in p.datasets:
        print(d.dataset_id)

# cmems_mod_med_wav_anfc_4.2km_PT1H-i
# cmems_mod_med_wav_anfc_4.2km_static
WAVE_DATASET_ID = "cmems_mod_med_wav_anfc_4.2km_PT1H-i"

Fetching catalogue 1: 100%|██████████| 2/2 [00:00<00:00,  2.27it/s]

cmems_mod_med_wav_anfc_4.2km_PT1H-i
cmems_mod_med_wav_anfc_4.2km_static


### get datasets

In [5]:
WAVE_VARS = ["VHM0", "VTPK", "VTM10", "VMDR",
             "VHM0_WW", "VTM01_WW", "VMDR_WW",
             "VHM0_SW1", "VTM01_SW1", "VMDR_SW1"]

In [6]:
# Download once to a local NetCDF. Tight bbox around your spots keeps it small.
copernicusmarine.subset(
    dataset_id=WAVE_DATASET_ID,
    variables=WAVE_VARS,
    start_datetime=START.strftime("%Y-%m-%dT%H:%M:%S"),
    end_datetime=END.strftime("%Y-%m-%dT%H:%M:%S"),
    output_filename="waves.nc",
    output_directory="data",
    overwrite=True,
    **BBOX,
)

# local in-mem file downloaded data
ds = xr.open_dataset("data/waves.nc").load()
# show all variables (will just be the ones filtered from my list)
# ds.data_vars

INFO - 2026-11-18T11:31:56Z - Selected dataset version: "202311"
INFO - 2026-11-18T11:31:56Z - Selected dataset part: "default"
100%|██████████| [06:33<00:00]  
INFO - 2026-11-18T11:38:31Z - Total size of the download: 204.41 MB.


In [7]:
# Extract a point (snaps to nearest sea cell, since coastal cells can be NaN)
def nearest_ocean_cell(da2d, lat, lon):
    lats, lons = np.meshgrid(da2d.latitude.values, da2d.longitude.values, indexing="ij")
    dist = np.hypot(lats - lat, (lons - lon) * np.cos(np.radians(lat)))
    dist = np.where(np.isnan(da2d.values), np.inf, dist)
    return np.unravel_index(dist.argmin(), dist.shape)

def wave_point(ds, lat, lon, variables):
    first = ds["VHM0"].isel(time=0).load()
    i, j = nearest_ocean_cell(first, lat, lon)
    df = ds[variables].isel(latitude=i, longitude=j).to_dataframe()
    df = df.drop(columns=[c for c in ("latitude", "longitude") if c in df.columns])
    df.index = pd.to_datetime(df.index).tz_localize("UTC").tz_convert(TZ)
    df.index.name = "time"
    return df


# Wind from Open-Meteo
def get_wind(lat, lon, days=8, models=None):
    params = dict(
        latitude=lat, longitude=lon,
        hourly="wind_speed_10m,wind_direction_10m,wind_gusts_10m",
        wind_speed_unit="kn", timezone="UTC", forecast_days=days,
    )
    if models:  # e.g. "icon_d2", "ecmwf_ifs025", "gfs_seamless"
        params["models"] = models
    r = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=30)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["hourly"])
    df["time"] = pd.to_datetime(df["time"], utc=True)
    return (df.set_index("time").tz_convert(TZ)
              .rename(columns={"wind_speed_10m": "wind_kn",
                               "wind_direction_10m": "wind_dir",
                               "wind_gusts_10m": "gust_kn"}))

# Merge and derive per-spot fields
def angdiff(a, b):
    """Signed smallest angle a-b in degrees, [-180, 180)."""
    return (a - b + 180) % 360 - 180

def build_frame(waves, wind, spot):
    df = waves.join(wind, how="inner")
    df["swell_off"] = angdiff(df["VMDR"], spot["facing"]).abs()
    df["in_window"] = df["swell_off"] <= spot["window"]

    w_off = angdiff(df["wind_dir"], spot["facing"]).abs()
    df["wind_rel"] = np.where(w_off >= 120, "offshore",
                      np.where(w_off <= 60, "onshore", "cross"))

    tm = df["VTM10"] if "VTM10" in df else df["VTPK"]
    if spot.get("use_wind_waves"):
        hs, per = df["VHM0"], tm
    else:
        hs = df["VHM0_SW1"].fillna(df["VHM0"])
        per = df["VTM01_SW1"].fillna(tm)
    # df["power_kw_m"] = 0.49 * df["VHM0"] ** 2 * tm     # deep-water power, relative use only
    df["power_kw_m"] = 0.49 * hs ** 2 * per
    df["swell_share"] = df.get("VHM0_SW1", np.nan) / df["VHM0"]  # swell vs wind-wave mix

    return df

In [8]:
def in_dataset(ds, lat, lon, margin=0.1):
    """True if the point lies inside the downloaded box (with a small margin)."""
    return (ds.latitude.min() + margin <= lat <= ds.latitude.max() - margin and
            ds.longitude.min() + margin <= lon <= ds.longitude.max() - margin)

frames = {}
for name, s in SPOTS.items():
    if not in_dataset(ds, s["lat"], s["lon"]):
        print(f"skip {name}: outside the downloaded box")
        continue
    w = wave_point(ds, s["lat"], s["lon"], WAVE_VARS)
    wi = get_wind(s["lat"], s["lon"])
    frames[name] = build_frame(w, wi, s)

frames["lagouvardos"][["VHM0", "VTPK", "VMDR", "wind_kn", "wind_rel", "in_window", "power_kw_m"]].round(1)

,VHM0,VTPK,VMDR,wind_kn,wind_rel,in_window,power_kw_m
time,,,,,,,
2026-11-18 07:00:00+02:00,1.3,5.2,255.5,4.1,onshore,True,1.6
2026-11-18 08:00:00+02:00,1.4,5.7,252.7,3.8,onshore,True,4.3
2026-11-18 09:00:00+02:00,1.5,6.3,251.6,5.2,onshore,True,5.0
2026-11-18 10:00:00+02:00,1.5,6.3,251.6,8.4,onshore,True,5.8
2026-11-18 11:00:00+02:00,1.6,6.9,251.8,9.6,onshore,True,6.3
...,...,...,...,...,...,...,...
2026-11-25 09:00:00+02:00,0.5,5.7,270.3,6.1,offshore,True,0.5
2026-11-25 10:00:00+02:00,0.5,5.2,269.1,6.1,offshore,True,0.4
2026-11-25 11:00:00+02:00,0.5,5.2,267.9,6.3,cross,True,0.4


In [9]:
frames

{'lagouvardos':                             VHM0   VTPK  VTM10        VMDR  VHM0_WW  VTM01_WW  \
 time                                                                            
 2026-11-18 07:00:00+02:00  1.271  5.209  4.810  255.469998    0.877     3.507   
 2026-11-18 08:00:00+02:00  1.385  5.730  5.217  252.729998    0.521     2.728   
 2026-11-18 09:00:00+02:00  1.467  6.303  5.513  251.579998    0.603     2.959   
 2026-11-18 10:00:00+02:00  1.534  6.303  5.734  251.559998    0.540     2.850   
 2026-11-18 11:00:00+02:00  1.557  6.934  5.893  251.849998    0.384     2.483   
 ...                          ...    ...    ...         ...      ...       ...   
 2026-11-25 09:00:00+02:00  0.494  5.730  5.181  270.329998    0.037     1.034   
 2026-11-25 10:00:00+02:00  0.472  5.209  5.149  269.059998    0.028     0.972   
 2026-11-25 11:00:00+02:00  0.451  5.209  5.107  267.949998    0.020     0.972   
 2026-11-25 12:00:00+02:00  0.432  5.209  5.065  266.859998    0.000     1.000   
 

## DUMP

In [10]:
import json

# spots (tuples become JSON arrays automatically)
json.dump(SPOTS, open("surf_engine/spots.json", "w"))

# forecasts: drop NaN rows, one list of hourly records per spot
# need = ["VHM0", "VTPK", "VMDR", "wind_kn", "wind_dir"]
need = ["VHM0", "VTPK", "VTM10", "VMDR", "wind_kn", "wind_dir", "power_kw_m"]
forecast = {
    name: json.loads(df.dropna(subset=need).reset_index().to_json(orient="records", date_format="iso"))
    for name, df in frames.items()
}
json.dump(forecast, open("surf_engine/forecast.json", "w"))

# TODO: Rust
run engine from python
get output here

# TODO: write to firestore
write to "spots" as manual BATCH
## TODO: determine data model
events: rust engine output
spots: manual spots
raw data??